In [ ]:
!pip install rasterio rioxarray xarray psycopg2-binary sqlalchemy

In [2]:
import rasterio
import rioxarray as rxr
import xarray as xr
from sqlalchemy import create_engine
print("Libraries are already available!")

Libraries are already available!


In [3]:
# NOAA GRIB2 direct URL using GDAL's virtual curl file system
url = "/vsicurl/https://tgftp.nws.noaa.gov/SL.us008001/ST.opnl/DF.gr2/DC.ndfd/AR.conus/VP.001-003/ds.pop12.bin"

# Open the multi-band dataset directly from the web
# rioxarray handles GRIB datasets seamlessly as a multi-dimensional data array
ds = xr.open_dataset(url, engine="rasterio")

print(ds)

<xarray.Dataset> Size: 118MB
Dimensions:      (band: 5, x: 2145, y: 1377)
Coordinates:
  * band         (band) int32 20B 1 2 3 4 5
  * x            (x) float64 17kB -2.763e+06 -2.761e+06 ... 2.679e+06 2.682e+06
  * y            (y) float64 11kB 3.231e+06 3.228e+06 ... -2.612e+05 -2.638e+05
    spatial_ref  int32 4B ...
Data variables:
    band_data    (band, y, x) float64 118MB ...


In [4]:
import rioxarray as rxr

url = "https://tgftp.nws.noaa.gov/SL.us008001/ST.opnl/DF.gr2/DC.ndfd/AR.conus/VP.001-003/ds.pop12.bin"

ds = rxr.open_rasterio(f"/vsicurl/{url}", masked=True)
print(ds)
print(ds.rio.crs)

<xarray.DataArray (band: 5, y: 1377, x: 2145)> Size: 118MB
[14768325 values with dtype=float64]
Coordinates:
  * band         (band) int32 20B 1 2 3 4 5
  * y            (y) float64 11kB 3.231e+06 3.228e+06 ... -2.612e+05 -2.638e+05
  * x            (x) float64 17kB -2.763e+06 -2.761e+06 ... 2.679e+06 2.682e+06
    spatial_ref  int32 4B 0
Attributes: (12/15)
    GRIB_UNIT:                           [%]
    GRIB_COMMENT:                        12 hr Prob of Precip > 0.01 In. [%]
    GRIB_ELEMENT:                        PoP12
    GRIB_SHORT_NAME:                     0-SFC
    GRIB_REF_TIME:                       1785101400
    GRIB_VALID_TIME:                     1785110400
    ...                                  ...
    GRIB_PDS_PDTN:                       9
    GRIB_PDS_TEMPLATE_NUMBERS:           1 8 2 0 0 0 255 255 0 0 0 0 0 1 0 0 ...
    GRIB_PDS_TEMPLATE_ASSEMBLED_VALUES:  1 8 2 0 0 255 255 0 0 1 0 0 255 -1 -...
    scale_factor:                        1.0
    add_offset:         

In [71]:
import rasterio

tif_path = r"E:\PostgreSQL\data1\pop121.tif"

with rasterio.open(tif_path) as src:
    print("Number of bands:", src.count)
    print("Band descriptions:", src.descriptions)
    print("Data type:", src.dtypes)
    print("NoData value:", src.nodata)
    print("CRS:", src.crs)
    print("Size:", src.width, "x", src.height)

Number of bands: 5
Band descriptions: ('0[-] SFC="Ground or water surface"', '0[-] SFC="Ground or water surface"', '0[-] SFC="Ground or water surface"', '0[-] SFC="Ground or water surface"', '0[-] SFC="Ground or water surface"')
Data type: ('uint8', 'uint8', 'uint8', 'uint8', 'uint8')
NoData value: None
CRS: PROJCS["unknown",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",25],PARAMETER["central_meridian",-95],PARAMETER["standard_parallel_1",25],PARAMETER["standard_parallel_2",25],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Size: 2145 x 1377


# SQL

In [5]:
import os
import subprocess
import psycopg2

# 1. Connect to PostgreSQL and enable PostGIS extensions
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="raster123",
    user="postgres",
    password="Postgresql321",  # <-- Replace with your real password
)

with conn.cursor() as cursor:
    cursor.execute("""
        CREATE EXTENSION IF NOT EXISTS postgis;
        CREATE EXTENSION IF NOT EXISTS postgis_raster;
    """)
    conn.commit()

conn.close()
print("PostGIS enabled successfully!")

Connected successfully!
Database Version: PostgreSQL 17.5 on x86_64-windows, compiled by msvc-19.44.35209, 64-bit


In [10]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",
        port="5432",
        database="raster123",
        user="postgres",
        password="Postgresql321",  # <-- Replace with your real password
    )
    with conn.cursor() as cursor:
        cursor.execute("SELECT version();")
        print("Connected successfully! DB Version:", cursor.fetchone()[0])
except Exception as e:
    print("Connection failed:", e)

Connected successfully! DB Version: PostgreSQL 17.5 on x86_64-windows, compiled by msvc-19.44.35209, 64-bit


In [11]:
with conn.cursor() as cursor:
    cursor.execute("""
        CREATE EXTENSION IF NOT EXISTS postgis;
        CREATE EXTENSION IF NOT EXISTS postgis_raster;
    """)
    conn.commit()
print("PostGIS extensions enabled successfully!")

PostGIS extensions enabled successfully!


In [12]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT schema_name 
        FROM information_schema.schemata;
    """)
    print("Available schemas:")
    for schema in cursor.fetchall():
        print(f" - {schema[0]}")

Available schemas:
 - public
 - weather
 - information_schema
 - pg_catalog
 - pg_toast


In [14]:
schema_name = "weather"  # Change to your target schema if needed

with conn.cursor() as cursor:
    cursor.execute(
        """
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = %s;
    """,
        (schema_name,),
    )
    print(f"Tables in schema '{schema_name}':")
    tables = cursor.fetchall()
    if tables:
        for table in tables:
            print(f" - {table[0]}")
    else:
        print(" (No tables found)")

Tables in schema 'weather':
 - pop12


## Tiling raster

In [46]:
import psycopg2

# Define everything once and assign it to one intuitive word: 'db'
db_raster = {
    "host": "localhost",
    "port": "5432",
    "database": "raster123",
    "user": "postgres",
    "password": "Postgresql321"
}

In [47]:
# Open connection using just the 'db' variable
conn = psycopg2.connect(**db_raster)

with conn.cursor() as cursor:
    cursor.execute("SELECT current_database();")
    print("Connected to:", cursor.fetchone()[0])

conn.close()

Connected to: raster123


In [41]:
!pip install folium

  Obtaining dependency information for folium from https://files.pythonhosted.org/packages/b5/a8/5f764f333204db0390362a4356d03a43626997f26818a0e9396f1b3bd8c9/folium-0.20.0-py2.py3-none-any.whl.metadata
  Obtaining dependency information for branca>=0.6.0 from https://files.pythonhosted.org/packages/7e/50/fc9680058e63161f2f63165b84c957a0df1415431104c408e8104a3a18ef/branca-0.8.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/113.4 kB ? eta -:--:--
   ---------------------------------------  112.6/113.4 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 113.4/113.4 kB 1.7 MB/s eta 0:00:00


## Download file, save it, customize projection (in SQL), tile data, and load as table

In [66]:
import os
import sys

print("Current Working Directory :", os.getcwd())
print("Python Executable Path    :", sys.executable)

Current Working Directory : C:\Users\srijal2023
Python Executable Path    : C:\Users\srijal2023\anaconda3\python.exe


In [81]:
import os
import rasterio

proj_dir = os.path.join(
    os.path.dirname(rasterio.__file__),
    "proj_data"
)

os.environ["PROJ_LIB"] = proj_dir
os.environ["PROJ_DATA"] = proj_dir

print(proj_dir)

C:\Users\srijal2023\anaconda3\Lib\site-packages\rasterio\proj_data


In [82]:
import psycopg2

conn = psycopg2.connect(
    dbname="raster123",
    user="postgres",
    password="Postgresql321",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

print("Connected:", conn.get_dsn_parameters()["dbname"])

Connected: raster123


In [94]:
import psycopg2

db_raster = psycopg2.connect(
    dbname="raster123",
    user="postgres",
    password="Postgresql321",
    host="localhost",
    port="5432"
)

cursor = db_raster.cursor()

print("Connected to raster123")

Connected to raster123


In [98]:
cursor.execute("""
DELETE FROM spatial_ref_sys
WHERE srid = 99000;
""")

db_raster.commit()

print("Existing SRID 99000 removed")

Existing SRID 99000 removed


In [99]:
srid = 99000

proj4text = (
    "+proj=lcc "
    "+lat_0=25 "
    "+lon_0=-95 "
    "+lat_1=25 "
    "+lat_2=25 "
    "+x_0=0 "
    "+y_0=0 "
    "+a=6371200 "
    "+b=6371200 "
    "+units=m "
    "+no_defs"
)

srtext = """
PROJCS["NOAA_NDFD_Lambert",
GEOGCS["Sphere",
DATUM["Custom",
SPHEROID["Sphere",6371200,0]],
PRIMEM["Greenwich",0],
UNIT["degree",0.0174532925199433]],
PROJECTION["Lambert_Conformal_Conic_2SP"],
PARAMETER["latitude_of_origin",25],
PARAMETER["central_meridian",-95],
PARAMETER["standard_parallel_1",25],
PARAMETER["standard_parallel_2",25],
PARAMETER["false_easting",0],
PARAMETER["false_northing",0],
UNIT["metre",1]]
"""

cursor.execute("""
INSERT INTO spatial_ref_sys
(
    srid,
    auth_name,
    auth_srid,
    proj4text,
    srtext
)
VALUES
(
    %s,
    'CUSTOM',
    %s,
    %s,
    %s
);
""",
(
    srid,
    srid,
    proj4text,
    srtext
))

db_raster.commit()

print("SRID 99000 created")

SRID 99000 created


In [97]:
db_raster.rollback()

print("Transaction reset")

Transaction reset


In [100]:
cursor.execute("""
SELECT srid, proj4text
FROM spatial_ref_sys
WHERE srid=99000;
""")

print(cursor.fetchone())

(99000, '+proj=lcc +lat_0=25 +lon_0=-95 +lat_1=25 +lat_2=25 +x_0=0 +y_0=0 +a=6371200 +b=6371200 +units=m +no_defs')


In [107]:
import rioxarray as rxr
import os

url = ("https://tgftp.nws.noaa.gov//SL.us008001/ST.opnl/DF.gr2/DC.ndfd//AR.conus/VP.001-003/ds.pop12.bin")


output_path = r"E:\PostgreSQL\data1\pop12_lcc.tif"


if os.path.exists(output_path):
    os.remove(output_path)


with rxr.open_rasterio(
    f"/vsicurl/{url}",
    masked=True
) as ds:

    print("Original CRS:")
    print(ds.rio.crs)

    raster = ds.fillna(255).astype("uint8")
    
    # Explicitly mark 255 as NoData
    raster.rio.write_nodata(255, inplace=True)

    raster.rio.to_raster(
        output_path,
        driver="GTiff",
        compress="DEFLATE",
        dtype="uint8",
        nodata=255
    )

print("Saved:", output_path)

Original CRS:
PROJCS["unnamed",GEOGCS["Coordinate System imported from GRIB file",DATUM["unnamed",SPHEROID["Sphere",6371200,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",25],PARAMETER["central_meridian",-95],PARAMETER["standard_parallel_1",25],PARAMETER["standard_parallel_2",25],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Saved: E:\PostgreSQL\data1\pop12_lcc.tif


In [101]:
import rasterio

with rasterio.open(output_path) as src:
    print(src.width, src.height)
    print(src.count)
    print(src.nodata)

2145 1377
5
None


In [104]:
db_raster = {
    "dbname": "raster123",
    "user": "postgres",
    "password": "Postgresql321",
    "host": "localhost",
    "port": "5432"
}

conn = psycopg2.connect(**db_raster)
cursor = conn.cursor()

env = os.environ.copy()
env["PGPASSWORD"] = db_raster["password"]

cmd = [
    psql,
    "-U", db_raster["user"],
    "-h", db_raster["host"],
    "-p", db_raster["port"],
    "-d", db_raster["dbname"]
]

In [108]:
import os
import subprocess

raster2pgsql = r"C:\Program Files\PostgreSQL\17\bin\raster2pgsql.exe"
psql = r"C:\Program Files\PostgreSQL\17\bin\psql.exe"

tif = r"E:\PostgreSQL\data1\pop12_lcc.tif"

# Password from db_raster
env = os.environ.copy()
env["PGPASSWORD"] = db_raster["password"]

cmd = (
    f'"{psql}" -U {db_raster["user"]} -d {db_raster["dbname"]} '
    f'-c "DROP TABLE IF EXISTS weather.pop12;" && '

    f'"{raster2pgsql}" '
    f'-s 99000 '
    f'-t 256x256 '
    f'-I '
    f'-C '
    f'-M '
    f'"{tif}" '
    f'weather.pop12 | '

    f'"{psql}" -U {db_raster["user"]} -d {db_raster["dbname"]}'
)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Return code:", process.returncode)

DROP TABLE
BEGIN
CREATE TABLE
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
INSERT 0 1
Processing 1/1: E:\PostgreSQL\data1\pop12_lcc.tif
INSERT 0 1
CREATE INDEX
ANALYZE
NOTICE:  Adding SRID constraint
NOTICE:  Adding scale-X constraint
NOTICE:  Adding scale-Y constraint
NOTICE:  Adding blocksize-X constraint
NOTICE:  Adding blocksize-Y constraint
NOTICE:  Adding alignment constraint
NOTICE:  Adding number of bands constraint
NOTICE:  Adding pixel type constraint
NOTICE:  Adding nodata value constraint
NOTICE:  Adding out-of-database constraint
NOTICE:  Adding maximum extent constraint
 addrasterconstraints 
----------------------
 t
(1 row)

COMMIT
VACU

In [73]:
import psycopg2

conn = psycopg2.connect(**db_raster)

with conn.cursor() as cursor:
    # 1. Overall dataset summary
    cursor.execute("""
        SELECT 
            COUNT(*) AS total_tiles,
            ST_SRID(rast) AS srid,
            MIN(ST_Width(rast)) AS min_width,
            MAX(ST_Width(rast)) AS max_width,
            MIN(ST_Height(rast)) AS min_height,
            MAX(ST_Height(rast)) AS max_height,
            MIN(ST_UpperLeftX(rast)) AS min_lon,
            MAX(ST_UpperLeftX(rast) + ST_Width(rast) * ST_ScaleX(rast)) AS max_lon,
            MIN(ST_UpperLeftY(rast) + ST_Height(rast) * ST_ScaleY(rast)) AS min_lat,
            MAX(ST_UpperLeftY(rast)) AS max_lat,
            MIN((ST_SummaryStats(rast, 1)).min) AS global_min,
            MAX((ST_SummaryStats(rast, 1)).max) AS global_max
        FROM weather.pop12
        GROUP BY ST_SRID(rast);
    """)
    overall = cursor.fetchone()
    
    print("==========================================")
    print("          OVERALL DATASET IDEA            ")
    print("==========================================")
    print(f"Total Tile Rows (Tiles): {overall[0]}")
    print(f"SRID (Projection)      : {overall[1]}")
    print(f"Tile Widths (Min-Max)  : {overall[2]} - {overall[3]} pixels")
    print(f"Tile Heights (Min-Max) : {overall[4]} - {overall[5]} pixels")
    print(f"Full Bounding Box      : Lon [{overall[6]:.4f}, {overall[7]:.4f}] | Lat [{overall[8]:.4f}, {overall[9]:.4f}]")
    print(f"Global Data Value Range: Min = {overall[10]} | Max = {overall[11]}")
    print("------------------------------------------\n")

    # 1.5. Identify any tiles with non-standard/different sizes
    cursor.execute("""
        SELECT rid, ST_Width(rast) AS width, ST_Height(rast) AS height
        FROM weather.pop12
        WHERE ST_Width(rast) <> (SELECT MODE() WITHIN GROUP (ORDER BY ST_Width(rast)) FROM weather.pop12)
           OR ST_Height(rast) <> (SELECT MODE() WITHIN GROUP (ORDER BY ST_Height(rast)) FROM weather.pop12);
    """)
    diff_tiles = cursor.fetchall()
    
    print("==========================================")
    print("        TILES WITH DIFFERENT SIZES        ")
    print("==========================================")
    if diff_tiles:
        for dt in diff_tiles:
            print(f"  -> Tile ID (RID): {dt[0]} | Non-Standard Size: {dt[1]}x{dt[2]} px")
    else:
        print("  -> All tiles share the exact same standard dimension.")
    print("------------------------------------------\n")

    # 2. Extreme Corner Tiles (Top-Left, Top-Right, Bottom-Left, Bottom-Right)
    print("==========================================")
    print("       EXTREME CORNER TILES IDEA          ")
    print("==========================================")
    
    corners = {
        "Top-Left (Min X, Max Y)": "ORDER BY ST_UpperLeftX(rast) ASC, ST_UpperLeftY(rast) DESC LIMIT 1",
        "Top-Right (Max X, Max Y)": "ORDER BY (ST_UpperLeftX(rast) + ST_Width(rast)*ST_ScaleX(rast)) DESC, ST_UpperLeftY(rast) DESC LIMIT 1",
        "Bottom-Left (Min X, Min Y)": "ORDER BY ST_UpperLeftX(rast) ASC, (ST_UpperLeftY(rast) + ST_Height(rast)*ST_ScaleY(rast)) ASC LIMIT 1",
        "Bottom-Right (Max X, Min Y)": "ORDER BY (ST_UpperLeftX(rast) + ST_Width(rast)*ST_ScaleX(rast)) DESC, (ST_UpperLeftY(rast) + ST_Height(rast)*ST_ScaleY(rast)) ASC LIMIT 1"
    }

    for label, order_clause in corners.items():
        cursor.execute(f"""
            SELECT 
                rid,
                ST_Width(rast) AS width,
                ST_Height(rast) AS height,
                ST_UpperLeftX(rast) AS ulx,
                ST_UpperLeftY(rast) AS uly,
                (ST_SummaryStats(rast, 1)).min AS min_val,
                (ST_SummaryStats(rast, 1)).max AS max_val
            FROM weather.pop12
            {order_clause};
        """)
        tile = cursor.fetchone()
        if tile:
            print(f"[{label}]")
            print(f"  -> Tile ID (RID): {tile[0]} | Size: {tile[1]}x{tile[2]} px")
            print(f"  -> Upper-Left   : X = {tile[3]:.4f}, Y = {tile[4]:.4f}")
            print(f"  -> Data Range   : Min = {tile[5]}, Max = {tile[6]}")
            print()

    # 3. Individual Random/Sample Tile Idea
    cursor.execute("""
        SELECT 
            rid,
            ST_Width(rast) AS width,
            ST_Height(rast) AS height,
            ST_UpperLeftX(rast) AS ulx,
            ST_UpperLeftY(rast) AS uly,
            (ST_SummaryStats(rast, 1)).min AS min_val,
            (ST_SummaryStats(rast, 1)).max AS max_val,
            (ST_SummaryStats(rast, 1)).mean AS mean_val
        FROM weather.pop12
        ORDER BY rid
        LIMIT 1;
    """)
    sample_tile = cursor.fetchone()
    
    print("==========================================")
    print("         SAMPLE TILE (RID = {})           ".format(sample_tile[0]))
    print("==========================================")
    print(f"Dimensions      : {sample_tile[1]} cols x {sample_tile[2]} rows")
    print(f"Top-Left Corner : X = {sample_tile[3]:.4f}, Y = {sample_tile[4]:.4f}")
    print(f"Tile Statistics : Min = {sample_tile[5]}, Max = {sample_tile[6]}, Mean = {sample_tile[7]:.2f}")
    print("==========================================")

conn.close()

          OVERALL DATASET IDEA            
Total Tile Rows (Tiles): 55
SRID (Projection)      : 99999
Tile Widths (Min-Max)  : 46 - 256 pixels
Tile Heights (Min-Max) : 204 - 256 pixels
Full Bounding Box      : Lon [-130.1229, -60.8598] | Lat [20.1788, 52.8170]
Global Data Value Range: Min = 0.0 | Max = 100.0
------------------------------------------

        TILES WITH DIFFERENT SIZES        
  -> Tile ID (RID): 11 | Non-Standard Size: 46x256 px
  -> Tile ID (RID): 22 | Non-Standard Size: 46x256 px
  -> Tile ID (RID): 33 | Non-Standard Size: 46x256 px
  -> Tile ID (RID): 44 | Non-Standard Size: 46x256 px
  -> Tile ID (RID): 45 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 46 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 47 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 48 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 49 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 50 | Non-Standard Size: 256x204 px
  -> Tile ID (RID): 51 | Non-Standard Size: 256x204 px
  

In [110]:
import folium
import json
import psycopg2

conn = psycopg2.connect(**db_raster)

with conn.cursor() as cursor:

    cursor.execute("""
        SELECT 
            rid,
            ST_Width(rast) AS width,
            ST_Height(rast) AS height,
            ST_AsGeoJSON(ST_Transform(ST_Envelope(rast), 4326)) AS geom_json,
            ST_Y(ST_Transform(ST_Centroid(ST_Envelope(rast)), 4326)) AS center_lat,
            ST_X(ST_Transform(ST_Centroid(ST_Envelope(rast)), 4326)) AS center_lon
        FROM weather.pop12;
    """)

    tiles = cursor.fetchall()

conn.close()


if not tiles:
    print("No tiles found in weather.pop12")

else:

    # Center map using first tile
    first_lat = tiles[0][4]
    first_lon = tiles[0][5]

    m = folium.Map(
        location=[first_lat, first_lon],
        zoom_start=5
    )

    for tile in tiles:

        rid, width, height, geom_json, lat, lon = tile

        geo_poly = json.loads(geom_json)

        folium.GeoJson(
            geo_poly,
            style_function=lambda x: {
                "color": "red",
                "weight": 2,
                "fillOpacity": 0.1,
            },
            tooltip=(
                f"RID: {rid}<br>"
                f"Size: {width} x {height} pixels"
            ),
        ).add_to(m)

#         folium.Marker(
#             location=[lat, lon],
#             popup=f"Tile ID: {rid}"
#         ).add_to(m)


    map_file = "tile_footprints_pop121.html"

    m.save(map_file)

    print(f"Saved: {map_file}")

Saved: tile_footprints_pop121.html


## Create GIF

In [113]:
import rasterio
import numpy as np
from PIL import Image

tif = r"E:\PostgreSQL\data1\pop12_lcc.tif"
gif_path = r"E:\PostgreSQL\data1\pop12_bands.gif"

frames = []

with rasterio.open(tif) as src:
    print("Bands:", src.count)
    nodata = src.nodata

    for band in range(1, src.count + 1):

        data = src.read(band)

        # Create mask for NoData
        mask = data == nodata

        # Normalize valid data only
        valid = data[~mask]

        img = np.zeros(data.shape, dtype="uint8")

        if valid.size > 0:
            img[~mask] = (
                (valid - valid.min()) /
                (valid.max() - valid.min()) * 200
            ).astype("uint8")

        # Set NoData background to light gray
        img[mask] = 230

        frames.append(Image.fromarray(img))


frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=1000,
    loop=0
)

print("Saved:", gif_path)

Bands: 5
Saved: E:\PostgreSQL\data1\pop12_bands.gif
